In [1]:
import json


class Cluster:
    def __init__(self, representative, members):
        self.representative = representative
        self.members = members

    def __repr__(self):
        return (
            f"Cluster(representative='{self.representative}', members={self.members})"
        )


def clean_name(name):
    return name.split("/")[-1].split(".")[0]


def load_clusters(array):
    result = []
    for obj in array:
        representative = clean_name(obj["representative"])
        members = list(map(clean_name, obj["members"]))
        result.append(Cluster(representative, members))
    return result


def load_all_clusters():
    for mode in ["approximate"]:
        for method in [
            "hierarchical",
            "affinity-propagation",
            "facility-location",
        ]:
            path = f"{mode}-{method}.json"
            with open(path) as f:
                data = json.load(f)
                yield (mode, method, load_clusters(data["clustering"]["clusters"]))


clustering = {}
for mode, method, clusters in load_all_clusters():
    clustering[(mode, method)] = clusters

print(f"Loaded {len(clustering)} clustering variants")
for (mode, method), clusters in clustering.items():
    print(f"  {mode} {method}: {len(clusters)} clusters")


Loaded 3 clustering variants
  approximate hierarchical: 219 clusters
  approximate affinity-propagation: 32 clusters
  approximate facility-location: 169 clusters


In [2]:
import pandas as pd

df = pd.read_csv("geometric_features.csv")
print(f"Dataset shape: {df.shape}")
print(f"Positive (double_tetrad=True): {df['double_tetrad'].sum()}")
print(f"Negative (double_tetrad=False): {(~df['double_tetrad']).sum()}")
df.head()


Dataset shape: (25917, 410)
Positive (double_tetrad=True): 21301
Negative (double_tetrad=False): 4616


,d01,d02,d03,d04,d05,d06,d07,d12,d13,d14,...,t3567,ts3567,ta3567,t4567,ts4567,ta4567,source_file,base_source_file,example_type,double_tetrad
0,5.281404,11.558636,14.495319,16.368533,16.042518,11.585704,8.681366,8.641264,11.495788,16.029153,...,0.995870,0.839232,0.543773,1.822365,0.968523,-0.248924,DT_139d-assembly1_000,DT_139d-assembly1_000,positive,True
1,5.828326,11.495788,14.831289,16.259659,15.661431,11.508291,7.724346,7.689616,11.544840,15.659610,...,0.902620,0.784953,0.619556,1.568627,0.999998,0.002169,DT_139d-assembly1_001,DT_139d-assembly1_001,positive,True
2,4.773614,11.544840,14.025507,16.342267,16.341829,11.572383,9.683914,9.653182,11.474521,16.340026,...,1.127830,0.903484,0.428621,2.130400,0.847465,-0.530850,DT_139d-assembly1_002,DT_139d-assembly1_002,positive,True
3,5.406934,11.277481,14.042228,16.265011,15.538241,11.305790,8.780451,9.025119,11.521149,16.335376,...,1.257319,0.951267,0.308369,2.040482,0.891711,-0.452606,DT_143d-assembly1_000,DT_143d-assembly1_000,positive,True
4,5.992458,11.521149,13.800741,16.462775,17.527265,11.788349,11.803337,10.610341,10.842241,15.857459,...,1.375624,0.981014,0.193935,2.589234,0.524696,-0.851290,DT_143d-assembly1_001,DT_143d-assembly1_001,positive,True


In [3]:
import itertools

N = 8

# Generate column names to drop (raw angle values, because we have the sine and cosine of these angles)
columns_to_drop = []

for i, j, k in itertools.combinations(range(N), 3):
    columns_to_drop.append(f"a{i}{j}{k}")

for i, j, k, l in itertools.combinations(range(N), 4):
    columns_to_drop.append(f"t{i}{j}{k}{l}")

df_filtered = df.drop(columns=columns_to_drop)
# Bookkeeping columns (not features)
meta_columns = ["source_file", "base_source_file", "example_type", "double_tetrad"]
print(f"Original columns: {len(df.columns)}")
print(f"Filtered columns: {len(df_filtered.columns)}")
print(
    f"Feature columns: {len(df_filtered.columns) - len(meta_columns)}  (excluding {meta_columns})"
)


Original columns: 410
Filtered columns: 284
Feature columns: 280  (excluding ['source_file', 'base_source_file', 'example_type', 'double_tetrad'])


In [4]:
from sklearn.model_selection import train_test_split

positive = df_filtered[df_filtered["double_tetrad"]]
negative = df_filtered[~df_filtered["double_tetrad"]]
splits = {}

for (mode, method), clusters in clustering.items():
    clusters.sort(key=lambda c: len(c.members))

    positive_test_names = []

    # The holdout fraction refers to distinct base motifs (phase variants
    # share their base motif's source_file, so counting rows would inflate
    # the threshold and pull all positives into the test set).
    n_base_positives = int((positive["example_type"] == "positive").sum())

    for cluster in clusters:
        positive_test_names.extend([cluster.representative] + cluster.members)
        if len(positive_test_names) >= 0.25 * n_base_positives:
            break

    positive_train = positive[~positive["source_file"].isin(positive_test_names)]
    positive_test = positive[positive["source_file"].isin(positive_test_names)]

    # Shuffled negatives share the geometry of their base motif, so they must
    # follow it into the same split (otherwise a shuffled copy of a test
    # motif could leak into training).  Phase positives inherit the base
    # motif's source_file, so the isin() split above already handles them.
    is_shuffled = negative["example_type"] == "shuffled_negative"
    shuffled = negative[is_shuffled]
    geometric = negative[~is_shuffled]
    shuffled_train = shuffled[~shuffled["base_source_file"].isin(positive_test_names)]
    shuffled_test = shuffled[shuffled["base_source_file"].isin(positive_test_names)]
    geometric_train, geometric_test = train_test_split(
        geometric, test_size=0.25, random_state=42
    )

    df_train = pd.concat([positive_train, shuffled_train, geometric_train])
    df_test = pd.concat([positive_test, shuffled_test, geometric_test])

    X_train = df_train.drop(columns=meta_columns)
    X_test = df_test.drop(columns=meta_columns)
    y_train = df_train["double_tetrad"]
    y_test = df_test["double_tetrad"]
    splits[(mode, method)] = (X_train, y_train, X_test, y_test)
    print(
        f"{mode} {method}: train={len(df_train)} (pos {len(positive_train)}, neg {len(shuffled_train) + len(geometric_train)}), test={len(df_test)} (pos {len(positive_test)}, neg {len(shuffled_test) + len(geometric_test)})"
    )


approximate hierarchical: train=19417 (pos 15961, neg 3456), test=6500 (pos 5340, neg 1160)
approximate affinity-propagation: train=19053 (pos 15621, neg 3432), test=6864 (pos 5680, neg 1184)
approximate facility-location: train=19401 (pos 15943, neg 3458), test=6516 (pos 5358, neg 1158)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Phase augmentation makes the training set positive-heavy (~12 correlated
# phase variants per motif); balance the classes where the estimator
# supports it.
classifiers = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "SVM": SVC(random_state=42, probability=True, class_weight="balanced"),
}


def evaluate_classifier(X_train, y_train, X_test, y_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for name, classifier in classifiers.items():
        classifier.fit(X_train_scaled, y_train)
        y_pred = classifier.predict(X_test_scaled)
        yield (
            name,
            classifier,
            scaler,
            classification_report(y_test, y_pred, output_dict=True),
        )


In [6]:
from pickle import dump

N = 8

for (mode, method), (X_train, y_train, X_test, y_test) in splits.items():
    print(f"\n=== Evaluating {mode} {method} ===")
    for name, clf, scaler, report in evaluate_classifier(
        X_train, y_train, X_test, y_test
    ):
        print(f"  Classifier: {name}")
        print(
            f"    accuracy={report['accuracy']:.4f}  "
            f"pos_precision={report['True']['precision']:.4f}  "
            f"pos_recall={report['True']['recall']:.4f}  "
            f"pos_f1={report['True']['f1-score']:.4f}"
        )

        model_name = name.lower().replace(" ", "-")
        payload = {
            "classifier_name": name,
            "classifier": clf,
            "scaler": scaler,
            "feature_columns": list(X_train.columns),
            "window_size": N,
            "positive_label": True,
            "split_mode": mode,
            "split_method": method,
        }

        with open(f"{mode}-{method}-{model_name}.pkl", "wb") as f:
            dump(payload, f)

print("\nDone. Trained and saved 5 classifiers x 3 splits = 15 model bundles.")



=== Evaluating approximate hierarchical ===
  Classifier: Naive Bayes
    accuracy=0.8728  pos_precision=0.9987  pos_recall=0.8463  pos_f1=0.9162
  Classifier: Logistic Regression
    accuracy=0.9397  pos_precision=0.9986  pos_recall=0.9279  pos_f1=0.9619
  Classifier: Decision Tree
    accuracy=0.8692  pos_precision=0.9967  pos_recall=0.8436  pos_f1=0.9138
  Classifier: Random Forest
    accuracy=0.9340  pos_precision=0.9988  pos_recall=0.9208  pos_f1=0.9582
  Classifier: SVM
    accuracy=0.9546  pos_precision=0.9988  pos_recall=0.9459  pos_f1=0.9716

=== Evaluating approximate affinity-propagation ===
  Classifier: Naive Bayes
    accuracy=0.9156  pos_precision=0.9982  pos_recall=0.8996  pos_f1=0.9464
  Classifier: Logistic Regression
    accuracy=0.7531  pos_precision=0.9987  pos_recall=0.7025  pos_f1=0.8248
  Classifier: Decision Tree
    accuracy=0.8014  pos_precision=0.9918  pos_recall=0.7664  pos_f1=0.8646
  Classifier: Random Forest
    accuracy=0.8122  pos_precision=0.9982  p